In [4]:
import sys
if '/home/ec2-user/sagemaker-pipe-template' not in sys.path:
    sys.path.insert(0, '/home/ec2-user/sagemaker-pipe-template')
import utils, boto3, paths
import pipeline.baseline as baseline
from sagemaker.core.helper.session_helper import Session
import pandas as pd
pd.set_option('display.max_colwidth', None)  # show full text in each cell
pd.set_option('display.max_columns', None)   # show all columns
pd.set_option('display.width', None)         # don't wrap

In [6]:
region = 'us-east-1'
model_package_group_name='abalone'
model_version=1
data_bucket='omm-test-bucket'
project_path = 'models/abalone'
account='088461143167'

role = utils.get_sm_service_role_arn()
boto_session=boto3.Session(region_name=region)
sm_client = boto_session.client('sagemaker', region_name=region)
sagemaker_session=Session(boto_session=boto_session)
p=paths.Paths(bucket_name='omm-test-bucket', project_name='abalone', model_prefix='abalone')

[06/15/26 20:26:41] INFO     Found credentials from IAM      ]8;id=9493386;file:///home/ec2-user/.local/lib/python3.9/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=9493387;file:///home/ec2-user/.local/lib/python3.9/site-packages/botocore/credentials.py#1171\1171]8;;\
                             Role: ec2-1                                        


In [ ]:
baseliner = baseline.Baseliner('abalone-1', p.data_dir, p.baseline_file, p.train_file, 'ml.m5.large', sagemaker_session)
baseliner.make_baseline_sets('rings', 'rings_prediction', target_type=float)

In [ ]:
def create_scheduled_data_quality_monitor(boto_session, role, name, deploy_type, schedule_expression, constraints_path, statistics_path, monitor_dir, endpoint_name=None, data_cature_dir=None, instance_type='ml.m5.large', volume_size_in_gb=20, max_runtime_in_seconds=1800):

    sm_client = boto_session.client('sagemaker', region_name='us-east-1')

    if deploy_type == 'realtime':
        monitoring_inputs=[{
            "EndpointInput": {
                "EndpointName": endpoint_name,
                "LocalPath": "/opt/ml/processing/input/endpoint"
            }
        }]
    else:
        monitoring_inputs=[{
            "BatchTransformInput": {
                "DataCapturedDestinationS3Uri": f"{data_cature_dir}/",
                "LocalPath": "/opt/ml/processing/input",
                "DatasetFormat": {
                    "Csv": {"Header": False}
                }
            }
        }]

    sm_client.create_monitoring_schedule(
        MonitoringScheduleName=name,
            MonitoringScheduleConfig={
            "ScheduleConfig": {
                "ScheduleExpression": schedule_expression# "cron(0 * ? * * *)"
            },
            "MonitoringJobDefinition": {
                "BaselineConfig": {
                    "ConstraintsResource": {"S3Uri": constraints_path},
                    "StatisticsResource": {"S3Uri": statistics_path}
                },
                "MonitoringInputs": monitoring_inputs,
                "MonitoringOutputConfig": {
                    "MonitoringOutputs": [{
                        "S3Output": {
                            "S3Uri": f'{monitor_dir}/reports',
                            "LocalPath": "/opt/ml/processing/output"
                        }
                    }]
                },
                "MonitoringResources": {
                    "ClusterConfig": {
                        "InstanceCount": 1,
                        "InstanceType": instance_type,
                        "VolumeSizeInGB": volume_size_in_gb
                    }
                },
                "MonitoringAppSpecification": {
                    "ImageUri": "156813124566.dkr.ecr.us-east-1.amazonaws.com/sagemaker-model-monitor-analyzer"
                },
                "RoleArn": role,
                "StoppingCondition": {"MaxRuntimeInSeconds": max_runtime_in_seconds}
            },
            "MonitoringType": "DataQuality"
        }
    )

In [ ]:
create_scheduled_data_quality_monitor(
    boto_session, 
    role, 
    name, 
    deploy_type, 
    schedule_expression, 
    constraints_path, 
    statistics_path, 
    monitor_dir, 
    endpoint_name=None, 
    data_cature_dir=None, 
    instance_type='ml.m5.large', 
    volume_size_in_gb=20, 
    max_runtime_in_seconds=1800
    )